**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · VALIDAÇÃO EXTERNA

# Módulo 00 — validação externa: reproduzir a série do InfoGripe (Fiocruz)

A pergunta deste caderno é a única que nenhuma checagem interna responde:
**o nosso Silver reproduz o que a Fiocruz publicou a partir dos mesmos
microdados?** Se a semana epidemiológica, a normalização e a definição de
caso estiverem certas, a série semanal nacional tem de bater — e o tamanho
do resíduo tem de ter explicação.

**Fontes, verificadas em 2026-08-31.** O alvo scriptável é
[`FluVigilanciaBR/data`](https://github.com/FluVigilanciaBR/data) (GPL-3.0,
PROCC/Fiocruz + FGV/EMAp + GT-Influenza/MS) — a série pública **congela na
temporada de 2019**, então a janela de comparação quantitativa é o ano
pré-pandêmico. O repositório autoritativo atual
(`gitlab.fiocruz.br/marcelo.gomes/infogripe`, apontado pelo próprio painel)
está com acesso anônimo bloqueado nesta data; o upgrade é criar conta no
GitLab da Fiocruz. Os boletins PDF continuam públicos e servem de
spot-check do presente.

**Este caderno não roda no CI de propósito**: depende de URL externa viva
(Git LFS do GitHub), e o CI deste repositório executa apenas
`*_walkthrough`/`*_internals`, que são auto-contidos no S3 pinado. Roda-se
manualmente; os outputs ficam commitados, como em todo o repositório.

[README do módulo](../README.md) · [o Prata](srag_silver_walkthrough.ipynb) · [o Prata, por dentro](srag_silver_internals.ipynb) · [`QUALITY.md`](../QUALITY.md)

---

In [1]:
%pip install -q pandas pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys
import tempfile
import urllib.request

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 120)

DATA = pathlib.Path.home() / "Documents/srag-data"
if not DATA.exists():
    DATA = pathlib.Path(tempfile.gettempdir()) / "srag-data"
    DATA.mkdir(parents=True, exist_ok=True)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "README.md").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "README.md").exists():
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP
import srag_30_silver as S

# --- os dois lados ---------------------------------------------------------
IG_URL = ("https://media.githubusercontent.com/media/FluVigilanciaBR/data/"
          "master/data/clean_data_epiweek-weekly-incidence_w_situation.csv")
ig_csv = DATA / "infogripe_clean_data.csv"
if not ig_csv.exists():
    urllib.request.urlretrieve(IG_URL, ig_csv)

bronze19 = DATA / "INFLUD19-26-06-2025.parquet"
if not bronze19.exists():
    urllib.request.urlretrieve(
        "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2019/"
        "INFLUD19-26-06-2025.parquet", bronze19)
print(f"InfoGripe: {SP.pt(ig_csv.stat().st_size / 1e6, 1)} MB   "
      f"Bronze 2019: {SP.pt(bronze19.stat().st_size / 1e6, 1)} MB")

InfoGripe: 83,0 MB   Bronze 2019: 3,2 MB


## §1 — A série publicada

Filtro documentado pelo esquema do arquivo: `dado = srag` (todas as
etiologias), `escala = casos` (contagem, não incidência), `sexo = Total`,
`Tipo = País`. A coluna `Situation` separa semanas consolidadas
(`stable`) das ainda incompletas na época do congelamento (`unknown`) —
a métrica estrita usa só as estáveis.

In [3]:
ig = pd.read_csv(ig_csv, low_memory=False)
pais = ig[(ig["dado"] == "srag") & (ig["escala"] == "casos")
          & (ig["sexo"] == "Total") & (ig["Tipo"] == "País")
          & (ig["epiyear"] == 2019)]
serie_ig = pais.set_index("epiweek")["SRAG"]
print(f"semanas publicadas: {len(serie_ig)}   total 2019: {SP.pt_int(serie_ig.sum())}")
print(f"situação: {dict(pais['Situation'].value_counts())}")

semanas publicadas: 48   total 2019: 39.018
situação: {'stable': np.int64(40), 'unknown': np.int64(8)}


## §2 — A reprodução, sob quatro definições

A definição de caso do InfoGripe pré-2021 **exigia febre**; a do script
atual do Ministério (`caso_srag_ms`) não. Em vez de assumir, testamos as
quatro candidatas e deixamos o ajuste identificar — é o mesmo método do
módulo inteiro: medir, não asseverar.

In [4]:
cols = ["DT_SIN_PRI", "HOSPITAL", "EVOLUCAO", "TOSSE", "GARGANTA", "DISPNEIA",
        "SATURACAO", "DESC_RESP", "FEBRE", "SG_UF_NOT"]
d = pq.read_table(bronze19, columns=cols).to_pandas()
n = {c: S.normalise(d[c], c) for c in cols}

def is1(c):
    return (n[c] == "1").fillna(False)

dt = S.parse_dates(n["DT_SIN_PRI"], "DT_SIN_PRI")
wk, yr = S.epiweek(dt), S.epiyear(dt)
sintomas = (is1("TOSSE") | is1("GARGANTA")) & (
    is1("DISPNEIA") | is1("SATURACAO") | is1("DESC_RESP"))
base = is1("HOSPITAL") | (n["EVOLUCAO"] == "2").fillna(False)
DEFS = {
    "caso_srag_ms (sem febre)": base & sintomas,
    "com febre (InfoGripe pré-2021)": base & sintomas & is1("FEBRE"),
    "hospitalizado cru": is1("HOSPITAL"),
    "todas as notificações": pd.Series(True, index=d.index),
}

em_2019 = yr == 2019
linhas = []
for nome, mask in DEFS.items():
    nossa = pd.Series(mask[em_2019]).groupby(wk[em_2019]).sum()
    j = pd.concat([serie_ig, nossa], axis=1, keys=["ig", "nos"]).dropna()
    dif = j["nos"] - j["ig"]
    linhas.append({
        "definição": nome,
        "total": int(mask[em_2019].sum()),
        "correlação": round(float(j["ig"].corr(j["nos"])), 4),
        "mediana |Δ|/publicado %": round(float((dif.abs() / j["ig"]).median()) * 100, 1),
        "Δ nas semanas comuns": int(dif.sum()),
    })
print(SP.tabela(pd.DataFrame(linhas),
                {"correlação": 4, "mediana |Δ|/publicado %": 1}))

                     definição  total correlação mediana |Δ|/publicado % Δ nas semanas comuns
      caso_srag_ms (sem febre) 44.078     0,9973                     8,2                3.552
com febre (InfoGripe pré-2021) 39.578     0,9979                     3,5                 -808
             hospitalizado cru 48.161     0,9962                    18,7                7.454
         todas as notificações 48.961     0,9961                    20,9                8.237


A definição **com febre** vence com folga: é a do InfoGripe de 2019,
identificada pelo ajuste. As outras três erram na direção e na escala
esperadas — sem febre sobra ~11%, hospitalizado cru sobra ~19%.

## §3 — A métrica estrita: só semanas estáveis, e por UF

In [5]:
melhor = base & sintomas & is1("FEBRE")
nossa = pd.Series(melhor[em_2019]).groupby(wk[em_2019]).sum()
estaveis = pais[pais["Situation"] == "stable"].set_index("epiweek")["SRAG"]
j = pd.concat([estaveis, nossa], axis=1, keys=["ig", "nos"]).dropna()
dif = j["nos"] - j["ig"]
print(f"semanas estáveis         : {len(j)}")
print(f"correlação               : {SP.pt(j['ig'].corr(j['nos']), 4)}")
print(f"mediana |Δ| por semana   : {SP.pt(float(dif.abs().median()), 0)} casos "
      f"({SP.pct(float((dif.abs() / j['ig']).median()), 1)} do publicado)")
print(f"Δ agregado               : {SP.pt(int(dif.sum()), 0, sinal=True)} "
      f"({SP.pct(dif.sum() / j['ig'].sum(), 1, sinal=True)})")

semanas estáveis         : 40
correlação               : 0,9997
mediana |Δ| por semana   : 28 casos (3,3% do publicado)
Δ agregado               : -1.204 (-3,3%)


In [6]:
uf_nome = {v: k for k, v in S.UF_CODES.items()}
por_uf_ig = (ig[(ig["dado"] == "srag") & (ig["escala"] == "casos")
                & (ig["sexo"] == "Total") & (ig["Tipo"] == "Estado")
                & (ig["epiyear"] == 2019) & (ig["Situation"] == "stable")]
             .assign(uf=lambda x: x["UF"].astype(int).astype(str).map(uf_nome))
             .groupby(["uf", "epiweek"])["SRAG"].sum())
nossa_uf = (pd.DataFrame({"uf": n["SG_UF_NOT"][em_2019], "wk": wk[em_2019],
                          "caso": melhor[em_2019]})
            .groupby(["uf", "wk"])["caso"].sum())
ju = pd.concat([por_uf_ig, nossa_uf], axis=1, keys=["ig", "nos"]).dropna()
res = (ju.groupby(level="uf")
         .apply(lambda g: pd.Series({
             "corr": g["ig"].corr(g["nos"]),
             "mediana |Δ| %": float(((g["nos"] - g["ig"]).abs()
                                     / g["ig"].clip(lower=1)).median()) * 100}))
       .sort_values("corr"))
print(f"UFs comparadas: {len(res)}")
print(SP.tabela(res.round(3).head(5), 3, index=True))
print("…")
print(SP.tabela(res.round(3).tail(3), 3, index=True))
print(f"\ncorrelação mediana entre UFs: {SP.pt(res['corr'].median(), 3)}")

UFs comparadas: 27
     corr mediana |Δ| %
uf                     
RR  0,915         0,000
MA  0,940         0,000
MT  0,966         0,000
AL  0,977         0,000
TO  0,986         0,000
…
     corr mediana |Δ| %
uf                     
MS  1,000         1,768
AM  1,000         0,000
AP  1,000         0,000

correlação mediana entre UFs: 0,997


## §4 — Leitura, resíduo e limites

- **A reprodução fecha.** Semana MMWR, normalização e lógica de sintomas
  reconstroem a série nacional que a Fiocruz publicou, com a definição
  identificada pelo ajuste — não haveria correlação ~0,998 com a semana
  errada (a ISO discorda do gabarito em 1 registro de cada 7).
- **O resíduo tem direção e explicação candidata**: comparamos a
  reexportação de 26/06/2025 com um congelamento de 2021 — notificações
  tardias, correções e exclusões entre as duas safras. O que o resíduo
  **não** pode esconder: erro de semana (testado à parte, 100,00%) ou de
  normalização (idem).
- **Limites declarados.** A série pública para em 2019; a validação
  pandêmica fica pendente do acesso ao GitLab da Fiocruz (conta a criar)
  ou do secundário `covid19br/sumario_SIVEP` (MIT, semana×município
  2020-06→2023-03, grupo adjacente — não é o pipeline InfoGripe). Os
  boletins PDF atuais (`agencia.fiocruz.br/.../Resumo_InfoGripe_*.pdf`)
  seguem públicos para spot-check de manchete.

O peso institucional que esta validação compra: o tratamento do módulo 00
não é uma leitura particular da base — reproduz, número a número na janela
disponível, o produto de vigilância da Fiocruz construído dos mesmos
microdados.